## 1、完成对Facebook数据的k近邻的机器学习，并通过网格搜索找到最佳的kneighbors

In [1]:
import pandas as pd

# 读取FBlocation文件夹下的train.csv
data = pd.read_csv("FBlocation/train.csv")

print(type(data))
print(data.shape)
# 预处理：去除有缺失值的行
data = data.dropna()

# 查看数据基本信息
print("删除缺失值后数据形状：", data.shape)
print("前5行预览：")
print(data.head())



<class 'pandas.core.frame.DataFrame'>
(29118021, 6)
删除缺失值后数据形状： (29118021, 6)
前5行预览：
   row_id       x       y  accuracy    time    place_id
0       0  0.7941  9.0809        54  470702  8523065625
1       1  5.9567  4.7968        13  186555  1757726713
2       2  8.3078  7.0407        74  322648  1137537235
3       3  7.3665  2.5165        65  704587  6567393236
4       4  4.0961  1.1307        31  472130  7440663949


In [2]:
# 1、缩小数据,查询数据,为了减少计算时间
data = data.query("x > 0.5 &  x < 1 & y > 1 & y < 2")
print(data.shape)

(151135, 6)


In [3]:
# 2、处理数据特征和标签
# 提取标签列（Label/Target）
y = data['place_id']  # 返回值类型：pandas.Series

# 删除不需要的列，保留特征列（Features）
X = data.drop(['row_id', 'place_id'], axis=1)  # 返回值类型：pandas.DataFrame
# axis=1表示删除列，axis=0表示删除行

# 处理time列，转换为datetime格式并提取时间特征
X['time'] = pd.to_datetime(X['time'], unit='s')  # 将秒级时间戳转为datetime类型
# unit='s'表示时间戳单位是秒

# 使用.dt访问器提取时间特征（返回值类型：pandas.Series）
X['day'] = X['time'].dt.day        # 提取日期中的天数（1-31）
X['hour'] = X['time'].dt.hour      # 提取小时（0-23）
X['weekday'] = X['time'].dt.weekday  # 提取星期几（0=周一，6=周日）

print(X)
print('-'*50)

# 删除原始time列，保留新的时间特征
X = X.drop(['time'], axis=1)

# 查看处理后的数据信息
print("特征矩阵形状：", X.shape)           # 返回值类型：tuple (行数, 列数)
print("标签向量形状：", y.shape)           # 返回值类型：tuple (行数,)
print("特征列：", X.columns.tolist())      # 返回值类型：list (Python列表)
print("前5行特征数据：")
print(X.head())                            # 返回值类型：pandas.DataFrame


               x       y  accuracy                time  day  hour  weekday
12        0.8829  1.3445        64 1970-01-07 15:34:48    7    15        2
172       0.7061  1.3806        52 1970-01-01 19:57:47    1    19        3
253       0.5305  1.8821        59 1970-01-04 23:01:42    4    23        6
601       0.9188  1.9335        56 1970-01-06 10:44:14    6    10        1
681       0.8689  1.9918        65 1970-01-01 05:50:46    1     5        3
...          ...     ...       ...                 ...  ...   ...      ...
29116687  0.8452  1.5151        99 1970-01-07 09:55:50    7     9        2
29116749  0.6940  1.7675        68 1970-01-05 10:49:12    5    10        0
29116753  0.9415  1.1621        64 1970-01-01 14:16:54    1    14        3
29116868  0.6481  1.0973       851 1970-01-09 04:10:40    9     4        4
29117677  0.9085  1.8273        72 1970-01-05 19:53:45    5    19        0

[151135 rows x 7 columns]
--------------------------------------------------
特征矩阵形状： (151135, 6)
标签

## 📚 代码中函数方法快速参考表

### 本代码段使用的所有函数方法汇总：

| 函数/方法 | 功能 | 语法示例 | 返回值类型 | 说明 |
|---------|------|---------|-----------|------|
| **`df['列名']`** | 提取单列数据 | `y = data['place_id']` | `pandas.Series` | 列选择，返回一维数据 |
| **`df.drop()`** | 删除列或行 | `X = data.drop(['列名'], axis=1)` | `pandas.DataFrame` | `axis=1`删除列，`axis=0`删除行 |
| **`pd.to_datetime()`** | 时间戳转换 | `pd.to_datetime(时间, unit='s')` | `pandas.Series` | `unit='s'`表示秒级时间戳 |
| **`.dt.day`** | 提取天数 | `X['time'].dt.day` | `pandas.Series` | 返回1-31的日期天数 |
| **`.dt.hour`** | 提取小时 | `X['time'].dt.hour` | `pandas.Series` | 返回0-23的小时数 |
| **`.dt.weekday`** | 提取星期几 | `X['time'].dt.weekday` | `pandas.Series` | 返回0-6（0=周一，6=周日） |
| **`.shape`** | 数据维度 | `X.shape` | `tuple` | 返回(行数, 列数)元组 |
| **`.columns`** | 列名索引 | `X.columns` | `pandas.Index` | 返回列名索引对象 |
| **`.tolist()`** | 转为列表 | `X.columns.tolist()` | `list` | 将索引/Series转为Python列表 |
| **`.head()`** | 查看前N行 | `X.head(n=5)` | `pandas.DataFrame` | 默认显示前5行 |

### 关键概念说明：

- **特征矩阵X（Features）**：用于训练模型的输入数据，通常是DataFrame类型
- **标签向量y（Label/Target）**：要预测的目标变量，通常是Series类型
- **特征工程（Feature Engineering）**：将原始数据转换为模型可用的特征的过程


In [4]:
# 3、统计标签数据（Label Statistics）
# 统计去重后的唯一值数目
unique_places = y.nunique()  # 返回值类型：int（整数）
# nunique()统计Series中唯一值的数量，不包括NaN
print(f"去重后的place_id数目：{unique_places}")

# 统计每个place_id的出现次数（频数统计）
place_counts = y.value_counts()  # 返回值类型：pandas.Series
# value_counts()返回每个唯一值及其出现次数，按出现次数降序排列
print(f"\n各place_id的重复数目：")
print(place_counts)

# 显示重复数目的统计信息（描述性统计）
print(f"\n重复数目统计：")
print(f"最多重复次数：{place_counts.max()}")      # 返回值类型：int，最大值
print(f"最少重复次数：{place_counts.min()}")      # 返回值类型：int，最小值
print(f"平均重复次数：{place_counts.mean():.2f}")  # 返回值类型：float，平均值（保留2位小数）


去重后的place_id数目：4233

各place_id的重复数目：
place_id
1317510067    1403
2374712395    1209
9551909561    1190
5241021697    1179
3496440943    1175
              ... 
9708599728       1
7179768014       1
3278278289       1
3269231486       1
7967579105       1
Name: count, Length: 4233, dtype: int64

重复数目统计：
最多重复次数：1403
最少重复次数：1
平均重复次数：35.70


## 📊 统计函数方法详解

### 本代码段使用的统计方法汇总：

| 函数/方法 | 功能 | 语法示例 | 返回值类型 | 说明 |
|---------|------|---------|-----------|------|
| **`.nunique()`** | 统计唯一值数量 | `y.nunique()` | `int` | 返回去重后的唯一值个数，不包括NaN |
| **`.value_counts()`** | 统计每个值的出现次数 | `y.value_counts()` | `pandas.Series` | 返回每个唯一值及其频数，按频数降序排列 |
| **`.max()`** | 最大值 | `place_counts.max()` | `int`或`float` | 返回Series中的最大值 |
| **`.min()`** | 最小值 | `place_counts.min()` | `int`或`float` | 返回Series中的最小值 |
| **`.mean()`** | 平均值 | `place_counts.mean()` | `float` | 返回Series中数值的平均值 |

### 方法详细说明：

#### 1. `.nunique()` - 唯一值计数
- **功能**：统计Series中唯一（不重复）值的数量
- **参数**：
  - `dropna=True`（默认）：不包括NaN值
  - `dropna=False`：包括NaN值
- **返回值**：整数（int），表示唯一值的个数
- **应用场景**：了解数据中有多少个不同的类别/标签

#### 2. `.value_counts()` - 频数统计
- **功能**：统计每个唯一值出现的次数
- **参数**：
  - `normalize=False`（默认）：返回频数（次数）
  - `normalize=True`：返回频率（比例，总和为1）
  - `sort=True`（默认）：按频数降序排列
  - `ascending=False`（默认）：降序排列
- **返回值**：pandas.Series，索引是唯一值，值是出现次数
- **应用场景**：了解数据分布、类别不平衡分析

#### 3. `.max()` / `.min()` - 最值统计
- **功能**：返回Series中的最大值/最小值
- **返回值**：数值类型（int或float）
- **应用场景**：了解数据的范围、极值

#### 4. `.mean()` - 平均值统计
- **功能**：计算Series中数值的平均值
- **返回值**：浮点数（float）
- **应用场景**：了解数据的中心趋势

### 关键概念：

- **唯一值（Unique Values）**：数据中不重复的值
- **频数（Frequency）**：某个值出现的次数
- **频率（Relative Frequency）**：某个值出现的比例（频数/总数）
- **描述性统计（Descriptive Statistics）**：描述数据特征的统计量（如均值、最值等）


In [5]:
# 4、数据过滤：去除只出现一次的place_id的样本（Data Filtering）
# 目的：只保留出现次数大于2的place_id，提高模型训练质量

# 步骤1：使用布尔索引筛选出重复次数大于2的place_id
# place_counts > 2 返回布尔Series，True表示该place_id出现次数>2
places_to_keep = place_counts[place_counts > 2].index  # 返回值类型：pandas.Index
# .index属性获取Series的索引（即place_id值），返回Index对象

# 步骤2：创建布尔掩码（Boolean Mask），标记哪些样本需要保留
mask = y.isin(places_to_keep)  # 返回值类型：pandas.Series (bool类型)
# isin()检查y中的每个值是否在places_to_keep中，返回True/False的Series
print(mask)

# 步骤3：使用布尔索引过滤数据（Boolean Indexing）
X_filtered = X[mask]  # 返回值类型：pandas.DataFrame
y_filtered = y[mask]  # 返回值类型：pandas.Series
# 布尔索引：只保留mask为True的行

# 步骤4：查看过滤效果
print(f"过滤前样本数量：{len(X)}")              # len()返回DataFrame的行数
print(f"过滤后样本数量：{len(X_filtered)}")      # len()返回DataFrame的行数
print(f"过滤前place_id数量：{y.nunique()}")      # 唯一值数量
print(f"过滤后place_id数量：{y_filtered.nunique()}")  # 唯一值数量

# 步骤5：更新X和y变量
X = X_filtered
y = y_filtered

# 步骤6：查看最终数据形状
print(f"\n过滤后的数据形状：")
print(f"特征矩阵形状：{X.shape}")    # 返回值类型：tuple (行数, 列数)
print(f"标签向量形状：{y.shape}")     # 返回值类型：tuple (行数,)


12          True
172         True
253         True
601         True
681         True
            ... 
29116687    True
29116749    True
29116753    True
29116868    True
29117677    True
Name: place_id, Length: 151135, dtype: bool
过滤前样本数量：151135
过滤后样本数量：148218
过滤前place_id数量：4233
过滤后place_id数量：1966

过滤后的数据形状：
特征矩阵形状：(148218, 6)
标签向量形状：(148218,)


## 🔍 数据过滤函数方法详解

### 本代码段使用的过滤方法汇总：

| 函数/方法/操作 | 功能 | 语法示例 | 返回值类型 | 说明 |
|-------------|------|---------|-----------|------|
| **布尔索引** | 根据条件筛选数据 | `series[series > 2]` | `pandas.Series` | 返回满足条件的行 |
| **`.index`** | 获取索引 | `series.index` | `pandas.Index` | 返回Series的索引对象 |
| **`.isin()`** | 检查值是否在列表中 | `y.isin([值1, 值2])` | `pandas.Series` (bool) | 返回布尔Series，True表示值在列表中 |
| **布尔索引** | 使用布尔掩码筛选 | `df[mask]` | `pandas.DataFrame` | 只保留mask为True的行 |
| **`len()`** | 获取长度/行数 | `len(df)` | `int` | 返回DataFrame的行数或列表长度 |

### 方法详细说明：

#### 1. 布尔索引（Boolean Indexing）- 条件筛选
- **功能**：根据条件表达式筛选数据
- **语法**：`series[条件表达式]` 或 `df[条件表达式]`
- **返回值**：满足条件的Series或DataFrame
- **工作原理**：
  - `place_counts > 2` 返回布尔Series（True/False）
  - `place_counts[place_counts > 2]` 只保留True对应的行
- **应用场景**：数据筛选、条件过滤

#### 2. `.index` 属性 - 获取索引
- **功能**：获取Series或DataFrame的索引
- **语法**：`series.index` 或 `df.index`
- **返回值**：`pandas.Index` 对象（类似列表的索引对象）
- **应用场景**：获取行标签、列名等

#### 3. `.isin()` 方法 - 成员检查
- **功能**：检查Series中的每个值是否在指定的列表/集合中
- **语法**：`series.isin([值1, 值2, ...])`
- **参数**：
  - 可以是列表、数组、Series或集合
- **返回值**：布尔Series（True表示值在列表中，False表示不在）
- **应用场景**：多值筛选、类别过滤

#### 4. 布尔索引（Boolean Indexing）- 掩码筛选
- **功能**：使用布尔Series作为掩码筛选数据
- **语法**：`df[布尔Series]` 或 `series[布尔Series]`
- **工作原理**：
  - 布尔Series的长度必须与DataFrame/Series的行数相同
  - 只保留布尔值为True的行
- **应用场景**：复杂条件筛选、数据过滤

#### 5. `len()` 函数 - 获取长度
- **功能**：返回对象的长度或元素个数
- **语法**：`len(对象)`
- **返回值**：整数（int）
- **对于不同对象**：
  - DataFrame：返回行数
  - Series：返回行数
  - 列表/字符串：返回元素个数
- **应用场景**：查看数据量、循环控制

### 关键概念：

- **布尔索引（Boolean Indexing）**：使用True/False值来选择数据的方法
- **掩码（Mask）**：布尔Series，用于标记哪些数据需要保留
- **条件筛选（Conditional Filtering）**：根据条件表达式筛选数据
- **数据过滤（Data Filtering）**：从数据集中选择满足条件的子集

### 代码执行流程：

1. **`place_counts[place_counts > 2]`** → 筛选出出现次数>2的place_id及其次数
2. **`.index`** → 提取这些place_id的值（Index对象）
3. **`y.isin(places_to_keep)`** → 创建布尔掩码，标记哪些样本需要保留
4. **`X[mask]` 和 `y[mask]`** → 使用掩码过滤数据，只保留True对应的行
5. **`len()`** → 查看过滤前后的数据量变化


In [6]:
# 5、数据集划分和KNN训练评估
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 划分训练集和测试集，测试集占25%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"训练集样本数量：{len(X_train)}")
print(f"测试集样本数量：{len(X_test)}")
print(f"训练集place_id数量：{y_train.nunique()}")
print(f"测试集place_id数量：{y_test.nunique()}")

# 数据标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  #fit_transform会在训练集计算均值和方差
X_test_scaled = scaler.transform(X_test)   #不会再次计算均值和方差，而是直接进行标准化

# 创建KNN分类器
knn = KNeighborsClassifier(n_neighbors=5)

# 训练模型
print("\n开始训练KNN模型...")
knn.fit(X_train_scaled, y_train)

# 计算准确率
accuracy = knn.score(X_test_scaled, y_test)
print(f"\nKNN模型准确率：{accuracy:.4f}")


训练集样本数量：111163
测试集样本数量：37055
训练集place_id数量：1966
测试集place_id数量：1966

开始训练KNN模型...

KNN模型准确率：0.3444


In [8]:
# 5、网格搜索（Grid Search）：找到最佳的KNN超参数（Hyperparameter Tuning）
from sklearn.model_selection import GridSearchCV

# 步骤1：定义参数网格（Parameter Grid）
# 参数网格是一个字典，键是参数名，值是要尝试的参数值列表
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],  # K值：尝试3, 5, 7, 9, 11
    'weights': ['uniform', 'distance']  # 权重方式
    # 'uniform': 每个邻居的权重相同（默认）
    # 'distance': 根据距离进行加权，距离越近权重越大，距离越远权重越小
}
# 总共会尝试 5 × 2 = 10 种参数组合

# 步骤2：创建KNN分类器（基础模型）
knn_grid = KNeighborsClassifier()  # 返回值类型：KNeighborsClassifier对象
# 不指定参数，让GridSearchCV来寻找最佳参数

# 步骤3：创建网格搜索对象（GridSearchCV）
grid_search = GridSearchCV(
    estimator=knn_grid,      # 要优化的模型
    param_grid=param_grid,   # 参数网格
    cv=5,                    # 交叉验证折数（3折交叉验证）
    scoring='accuracy',      # 评估指标（准确率）
    n_jobs=-1,              # 并行任务数（-1表示使用所有CPU核心）
    verbose=1                # 详细输出级别（1表示显示进度）
)
# 返回值类型：GridSearchCV对象

# 步骤4：执行网格搜索（训练所有参数组合）
print("开始网格搜索...")
grid_search.fit(X_train_scaled, y_train)  # 返回值类型：GridSearchCV对象（已训练）
# fit()方法会：
# 1. 对每种参数组合进行交叉验证
# 2. 计算平均得分
# 3. 选择得分最高的参数组合

# 步骤5：输出最佳参数和最佳得分
print(f"\n最佳参数：{grid_search.best_params_}")  # 返回值类型：dict（字典）
print(f"最佳交叉验证得分：{grid_search.best_score_:.4f}")  # 返回值类型：float（浮点数）
# best_params_：最佳参数组合（字典）
# best_score_：最佳交叉验证得分（浮点数）

# 步骤6：使用最佳参数在测试集上评估
best_knn = grid_search.best_estimator_  # 返回值类型：KNeighborsClassifier对象
# best_estimator_：使用最佳参数训练好的模型
test_accuracy = best_knn.score(X_test_scaled, y_test)  # 返回值类型：float
# score()方法：计算模型在测试集上的准确率
print(f"最佳模型在测试集上的准确率：{test_accuracy:.4f}")

# 步骤7：显示所有参数组合的详细结果
print("\n所有参数组合的结果：")
results = grid_search.cv_results_  # 返回值类型：dict（字典）
# cv_results_：包含所有参数组合的详细交叉验证结果
print(results)
# 遍历所有参数组合，显示每个组合的平均得分和标准差
for i in range(len(results['params'])):  # range()返回整数序列，len()返回列表长度
    print(f"n_neighbors={results['params'][i]['n_neighbors']}: "
          f"平均得分={results['mean_test_score'][i]:.4f} "
          f"(±{results['std_test_score'][i]:.4f})")
    # results['params']：所有参数组合的列表
    # results['mean_test_score']：每个参数组合的平均交叉验证得分
    # results['std_test_score']：每个参数组合的得分标准差


开始网格搜索...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


d:\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



最佳参数：{'n_neighbors': 7, 'weights': 'distance'}
最佳交叉验证得分：0.3544
最佳模型在测试集上的准确率：0.3687

所有参数组合的结果：
{'mean_fit_time': array([0.19668231, 0.25332527, 0.27571149, 0.34769888, 0.40314059,
       0.36031647, 0.35448351, 0.50158482, 0.33513875, 0.30831661]), 'std_fit_time': array([0.00683436, 0.02507693, 0.0259364 , 0.05623427, 0.0946016 ,
       0.07767083, 0.01641432, 0.10255449, 0.07011215, 0.01536873]), 'mean_score_time': array([1.20598826, 3.16766529, 1.55671644, 4.35332685, 2.05095081,
       5.76208181, 1.90698228, 6.04908886, 1.84248424, 6.14121737]), 'std_score_time': array([0.13566796, 0.43948166, 0.20628719, 0.18107633, 0.19611947,
       0.37581928, 0.04870232, 0.17873016, 0.06864312, 0.1301693 ]), 'param_n_neighbors': masked_array(data=[3, 3, 5, 5, 7, 7, 9, 9, 11, 11],
             mask=[False, False, False, False, False, False, False, False,
                   False, False],
       fill_value=999999), 'param_weights': masked_array(data=['uniform', 'distance', 'uniform', 'distance

## 🔍 网格搜索函数方法详解

### 本代码段使用的网格搜索方法汇总：

| 函数/方法/属性 | 功能 | 语法示例 | 返回值类型 | 说明 |
|-------------|------|---------|-----------|------|
| **`GridSearchCV()`** | 网格搜索交叉验证 | `GridSearchCV(estimator, param_grid, cv)` | `GridSearchCV对象` | 自动搜索最佳超参数 |
| **`KNeighborsClassifier()`** | 创建KNN分类器 | `KNeighborsClassifier()` | `KNeighborsClassifier对象` | K近邻分类器 |
| **`.fit()`** | 训练模型/执行搜索 | `grid_search.fit(X, y)` | `GridSearchCV对象` | 执行网格搜索和交叉验证 |
| **`.best_params_`** | 最佳参数 | `grid_search.best_params_` | `dict` | 返回最佳参数组合（字典） |
| **`.best_score_`** | 最佳得分 | `grid_search.best_score_` | `float` | 返回最佳交叉验证得分 |
| **`.best_estimator_`** | 最佳模型 | `grid_search.best_estimator_` | `模型对象` | 使用最佳参数训练好的模型 |
| **`.score()`** | 模型评估 | `model.score(X, y)` | `float` | 计算模型准确率 |
| **`.cv_results_`** | 所有结果 | `grid_search.cv_results_` | `dict` | 包含所有参数组合的详细结果 |
| **`len()`** | 获取长度 | `len(list)` | `int` | 返回列表/数组的长度 |
| **`range()`** | 生成整数序列 | `range(n)` | `range对象` | 返回0到n-1的整数序列 |

### 方法详细说明：

#### 1. `GridSearchCV()` - 网格搜索交叉验证
- **功能**：自动搜索最佳超参数组合
- **语法**：`GridSearchCV(estimator, param_grid, cv, scoring, n_jobs, verbose)`
- **主要参数**：
  - `estimator`：要优化的模型（如KNeighborsClassifier）
  - `param_grid`：参数网格（字典），键是参数名，值是要尝试的参数值列表
  - `cv`：交叉验证折数（如3表示3折交叉验证）
  - `scoring`：评估指标（如'accuracy'表示准确率）
  - `n_jobs`：并行任务数（-1表示使用所有CPU核心）
  - `verbose`：详细输出级别（1表示显示进度）
- **返回值**：GridSearchCV对象
- **工作原理**：
  1. 遍历参数网格中的所有参数组合
  2. 对每种组合进行交叉验证
  3. 计算平均得分
  4. 选择得分最高的参数组合

#### 2. `KNeighborsClassifier()` - K近邻分类器
- **功能**：创建KNN分类器
- **语法**：`KNeighborsClassifier(n_neighbors, weights, ...)`
- **主要参数**：
  - `n_neighbors`：K值（邻居数量），默认5
  - `weights`：权重方式
    - `'uniform'`：每个邻居权重相同（默认）
    - `'distance'`：根据距离加权，距离越近权重越大
- **返回值**：KNeighborsClassifier对象

#### 3. `.fit()` - 训练模型
- **功能**：训练模型或执行网格搜索
- **语法**：`grid_search.fit(X, y)`
- **参数**：
  - `X`：特征矩阵（训练数据）
  - `y`：标签向量（目标变量）
- **返回值**：GridSearchCV对象（已训练）
- **执行过程**：对所有参数组合进行交叉验证训练

#### 4. `.best_params_` - 最佳参数
- **功能**：获取最佳参数组合
- **语法**：`grid_search.best_params_`
- **返回值**：字典（dict），键是参数名，值是最佳参数值
- **示例**：`{'n_neighbors': 5, 'weights': 'distance'}`

#### 5. `.best_score_` - 最佳得分
- **功能**：获取最佳交叉验证得分
- **语法**：`grid_search.best_score_`
- **返回值**：浮点数（float），表示最佳交叉验证得分
- **说明**：这是所有参数组合中得分最高的

#### 6. `.best_estimator_` - 最佳模型
- **功能**：获取使用最佳参数训练好的模型
- **语法**：`grid_search.best_estimator_`
- **返回值**：模型对象（如KNeighborsClassifier）
- **说明**：可以直接用于预测，无需重新训练

#### 7. `.score()` - 模型评估
- **功能**：计算模型在给定数据上的准确率
- **语法**：`model.score(X, y)`
- **参数**：
  - `X`：特征矩阵
  - `y`：真实标签
- **返回值**：浮点数（float），表示准确率（0-1之间）

#### 8. `.cv_results_` - 所有结果
- **功能**：获取所有参数组合的详细交叉验证结果
- **语法**：`grid_search.cv_results_`
- **返回值**：字典（dict），包含：
  - `'params'`：所有参数组合的列表
  - `'mean_test_score'`：每个参数组合的平均得分
  - `'std_test_score'`：每个参数组合的得分标准差
  - `'rank_test_score'`：得分排名
- **应用场景**：分析不同参数组合的表现

#### 9. `len()` - 获取长度
- **功能**：返回对象的长度或元素个数
- **语法**：`len(对象)`
- **返回值**：整数（int）
- **应用场景**：获取列表长度、循环控制

#### 10. `range()` - 生成整数序列
- **功能**：生成整数序列
- **语法**：`range(start, stop, step)` 或 `range(stop)`
- **返回值**：range对象（可迭代）
- **示例**：
  - `range(5)` → 0, 1, 2, 3, 4
  - `range(1, 5)` → 1, 2, 3, 4
  - `range(0, 10, 2)` → 0, 2, 4, 6, 8

### 关键概念：

- **超参数（Hyperparameters）**：模型训练前需要设置的参数（如K值、权重方式）
- **网格搜索（Grid Search）**：穷举搜索所有参数组合的方法
- **交叉验证（Cross Validation）**：将数据分成K折，轮流作为训练集和验证集
- **参数网格（Parameter Grid）**：要搜索的参数及其可能值的组合
- **最佳参数（Best Parameters）**：在所有尝试的参数组合中得分最高的组合

### 网格搜索工作流程：

1. **定义参数网格** → 指定要尝试的参数值
2. **创建GridSearchCV对象** → 设置模型、参数网格、交叉验证折数等
3. **执行fit()** → 对所有参数组合进行交叉验证
4. **获取最佳参数** → 使用`best_params_`和`best_score_`
5. **使用最佳模型** → 使用`best_estimator_`进行预测


## 2、完成朴素贝叶斯公式的理解，针对上课讲的科技类，金融类的样例，来解释你对朴素贝叶斯的理解 

朴素贝叶斯通过在类别条件下假设特征相互独立，将联合概率分解为条件概率的乘积，并结合先验概率进行比较，从而实现高效且可解释的分类决策，尤其适用于文本分类任务。

人话版：贝叶斯公式就是执果索因，在结果发生的条件下去看是哪个原因造成了结果发生，选出概率最大的那个原因